In [ ]:
import numpy as np
import math
import random


def entropy(rv):
    total = len(rv)
    _, cnts = np.unique(rv, return_counts=True)
    entropy = 0
    for cnt in cnts:
        p_i = cnt / total
        entropy += -(p_i) * math.log2(p_i)
    return entropy


class Node:
    def __init__(self, *, terminal=False, label=None, next=None, result=None):
        self.terminal = terminal
        self.label = label
        self.next = {} if next is None else next
        self.result = result


class DecisionTreeClassifier:
    def __init__(self):
        pass

    def construct(self, X, y, possible_cols):
        if len(possible_cols) == 0:
            return Node(terminal=True, result=y[0])  # TODO: improve heuristic?

        S_entropy = entropy(y)

        if S_entropy == 0:
            return Node(terminal=True, result=y[0])  # TODO: improve heuristic?

        S = len(y)
        max_idx = -1
        max_info_gain = -math.inf

        for col in possible_cols:
            rv = X[:, col]
            info_gain = S_entropy
            for val in np.unique(rv):
                mask = rv == val
                info_gain -= len(y[mask]) * entropy(y[mask]) / S
            if info_gain > max_info_gain:
                max_idx = col
                max_info_gain = info_gain

        winner_rv = X[:, max_idx]
        curr = Node(terminal=False, label=max_idx)
        possible_cols.remove(max_idx)
        for val in np.unique(winner_rv):
            mask = winner_rv == val
            X_new = X[mask]
            y_new = y[mask]
            curr.next[val] = self.construct(X_new, y_new, possible_cols.copy())
        return curr

    def fit(self, X, y):
        # we can pre-process here
        self.root = self.construct(X, y, [i for i in range(X.shape[1])])
        pass

    def classify(self, X):
        y = []
        for row in X:
            curr = self.root
            while not curr.terminal:
                val = row[curr.label]
                if not val in curr.next:
                    # TODO: improve the heuristic
                    val = random.choice(
                        list(iter(curr.next.keys()))
                    )  # INVARIANT: atleast has one value
                curr = curr.next[val]
            y.append(curr.result)
        return y

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

df = pd.read_csv("mushroom.csv")


encoders = {}

for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le


X = df.drop(columns=["class"]).to_numpy()
y = df["class"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)


my_tree = DecisionTreeClassifier()
my_tree.fit(X_train, y_train)
my_pred = my_tree.classify(X_test)


sk_tree = SklearnDecisionTree(
    random_state=42,
)

sk_tree.fit(X_train, y_train)
sk_pred = sk_tree.predict(X_test)


def print_metrics(name, y_true, y_pred):
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Accuracy : {accuracy_score(y_true, y_pred)}")
    print(f"Precision: {precision_score(y_true, y_pred)}")
    print(f"Recall   : {recall_score(y_true, y_pred)}")
    print(f"F1 Score : {f1_score(y_true, y_pred)}")

print_metrics("My ID3", y_test, my_pred)
print_metrics("Scikit-Learn", y_test, sk_pred)


My ID3
------
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Scikit-Learn
------------
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
